# 11 — Error analysis from unified predictions

Read the two files generated by notebook 10, sample error-analysis cases with seed 42, include the continuous toxicity score, and compare the BERT-base versus HateBERT A/B/C/D groups.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next(
    path for path in (Path.cwd(), Path.cwd().parent)
    if (path / "data/raw/train.csv").exists()
)
RESULTS = ROOT / "results"

FINAL = RESULTS / "error_analysis_full"
TEST = ROOT / "data/splits/full/test.csv"
FULL_PREDICTIONS = FINAL / "all_model_predictions_full_data.csv"
FIXED_PREDICTIONS = FINAL / "all_model_predictions_fixed_200k.csv"
MODELS = ["LR", "SVM", "DistilBERT", "BERT", "HateBERT"]
EXPECTED_TEST_ROWS = 178083
SAMPLE_SEED = 42


def load_aligned_predictions(path):
    assert path.exists(), f"Run notebook 10 first; missing: {path}"
    frame = pd.read_csv(path)
    required = {"id", "text", "true_label"} | {f"{model}_pred" for model in MODELS}
    assert required.issubset(frame.columns), sorted(required - set(frame.columns))
    assert len(frame) == EXPECTED_TEST_ROWS and frame["id"].is_unique
    canonical = pd.read_csv(TEST, usecols=["id", "comment_text", "label", "target"])
    assert set(frame["id"]) == set(canonical["id"])
    aligned = frame.set_index("id").reindex(canonical["id"])
    canonical = canonical.set_index("id")
    assert np.array_equal(
        aligned["true_label"].astype("int8").to_numpy(),
        canonical["label"].astype("int8").to_numpy(),
    )
    assert np.array_equal(
        aligned["text"].fillna("").astype(str).to_numpy(),
        canonical["comment_text"].fillna("").astype(str).to_numpy(),
    )
    aligned["continuous_toxicity_score"] = canonical["target"].to_numpy()
    return aligned.reset_index()


def add_error_flags(frame):
    frame = frame.copy()
    for model in MODELS:
        frame[f"{model}_error"] = (
            frame[f"{model}_pred"].astype("int8") != frame["true_label"].astype("int8")
        )
    return frame


def make_error_pool(frame):
    error_columns = [f"{model}_error" for model in MODELS]
    return frame.loc[frame[error_columns].any(axis=1)].copy()


full_predictions = add_error_flags(load_aligned_predictions(FULL_PREDICTIONS))
fixed_predictions = add_error_flags(load_aligned_predictions(FIXED_PREDICTIONS))
full_error_pool = make_error_pool(full_predictions)
fixed_error_pool = make_error_pool(fixed_predictions)
print("Full-data error pool:", len(full_error_pool))
print("Fixed-200k error pool:", len(fixed_error_pool))


def wrong_models(row):
    return ", ".join(
        model for model in MODELS
        if int(row[f"{model}_pred"]) != int(row["true_label"])
    )


group_masks = {
    "A": (
        (full_predictions["true_label"] == 1)
        & (full_predictions["BERT_pred"] == 0)
        & (full_predictions["HateBERT_pred"] == 1)
    ),
    "B": (
        (full_predictions["true_label"] == 1)
        & (full_predictions["BERT_pred"] == 1)
        & (full_predictions["HateBERT_pred"] == 0)
    ),
    "C": (
        (full_predictions["true_label"] == 0)
        & (full_predictions["BERT_pred"] == 0)
        & (full_predictions["HateBERT_pred"] == 1)
    ),
    "D": (
        (full_predictions["true_label"] == 0)
        & (full_predictions["BERT_pred"] == 1)
        & (full_predictions["HateBERT_pred"] == 0)
    ),
}

group_pools = {
    group: full_predictions.loc[mask].copy()
    for group, mask in group_masks.items()
}
group_sizes = {name: len(frame) for name, frame in group_pools.items()}
assert all(size >= 50 for size in group_sizes.values())

targeted = pd.concat(
    [
        pool.sample(n=50, random_state=SAMPLE_SEED).assign(group=group)
        for group, pool in group_pools.items()
    ],
    ignore_index=True,
)
targeted_ids = set(targeted["id"])
general = full_error_pool.loc[~full_error_pool["id"].isin(targeted_ids)].sample(
    n=100, random_state=SAMPLE_SEED
).copy()
general["group"] = "General"
general["error_type"] = general.apply(wrong_models, axis=1)

targeted["error_type"] = targeted.apply(wrong_models, axis=1)
sample_300 = pd.concat([general, targeted], ignore_index=True)
assert len(sample_300) == 300 and sample_300["id"].is_unique
assert sample_300["continuous_toxicity_score"].between(0, 1).all()

output_columns = [
    "id", "text", "true_label", "continuous_toxicity_score",
    "group", "error_type", *[f"{model}_pred" for model in MODELS],
]
sample_300 = sample_300[output_columns]
SAMPLE_OUTPUT = FINAL / "error_analysis_sample_300.csv"
SAMPLE_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
sample_300.to_csv(SAMPLE_OUTPUT, index=False)
print("Generated sample table:", SAMPLE_OUTPUT)
print("Sampled review cases:", len(general), "general +", len(targeted), "targeted")

In [ ]:
general_counts = general["error_type"].value_counts().rename_axis("error_type").reset_index(name="n")
display(general_counts)

## Model-error combinations

In [ ]:
general_predictions = general.copy()
display(general_predictions[["id", "text", "error_type", "continuous_toxicity_score"]])

## BERT-base versus HateBERT sampled groups

A and D are cases where HateBERT corrects BERT-base. B and C are cases where HateBERT introduces an error. Each group contains 50 cases sampled in this notebook with random seed 42.

In [ ]:
display(targeted[["id", "group", "text", "continuous_toxicity_score"]])

## Targeted sample summary

This section summarises the four BERT-base/HateBERT groups sampled above. Each row includes the model predictions and continuous toxicity score.

In [ ]:
group_table = pd.DataFrame({
    "group": ["A", "B", "C", "D"],
    "meaning": [
        "HateBERT corrects a BERT-base false negative",
        "HateBERT introduces a false negative",
        "HateBERT introduces a false positive",
        "HateBERT corrects a BERT-base false positive",
    ],
    "pool_n": [group_sizes[name] for name in ["A", "B", "C", "D"]],
    "sampled_n": [50, 50, 50, 50],
})
display(group_table)

## Continuous toxicity score analysis

In [ ]:
sample_from_file = pd.read_csv(SAMPLE_OUTPUT)
targeted_from_file = sample_from_file.loc[
    sample_from_file["group"].isin(["A", "B", "C", "D"])
].copy()
assert len(targeted_from_file) == 200

score_summary = (
    targeted_from_file.groupby("group")["continuous_toxicity_score"]
    .agg(n="count", mean="mean", median="median", minimum="min", maximum="max")
    .reset_index()
)
targeted_from_file["score_band"] = pd.cut(
    targeted_from_file["continuous_toxicity_score"],
    bins=[-np.inf, 0.5, 0.7, np.inf],
    labels=["below 0.5", "0.5 to 0.69", "0.7 or above"],
    right=False,
)
band_summary = pd.crosstab(targeted_from_file["group"], targeted_from_file["score_band"])
display(score_summary)
display(band_summary)

for group in ["C", "D"]:
    scores = targeted_from_file.loc[
        targeted_from_file["group"] == group,
        "continuous_toxicity_score"
    ]
    proportion = ((scores >= 0.30) & (scores < 0.50)).mean()
    print(
        group,
        "median =", scores.median(),
        "proportion_0.30_0.499 =", proportion,
    )

,group,n,mean,median,minimum,maximum
0,A,50,0.637894,0.6,0.5,1.000000
1,B,50,0.614667,0.6,0.5,1.000000
2,C,50,0.238184,0.3,0.0,0.426230
3,D,50,0.245243,0.3,0.0,0.471429


score_band,below 0.5,0.5 to 0.69,0.7 or above
group,,,
A,0,30,20
B,0,34,16
C,50,0,0
D,50,0,0


C median = 0.3 proportion_0.30_0.499 = 0.52
D median = 0.3 proportion_0.30_0.499 = 0.52
